# Explore the collected data

This notebook reads `data/warehouse.duckdb`, which is built from the raw archive. To refresh it:

1. Copy the raw archive from the VPS (over Tailscale):
   `rsync -a kac3pro@100.79.225.110:paper-trader/data/raw/ data/raw/`
2. Rebuild the warehouse: `uv run python -m paper_trader.apps.build_db`
3. Re-run the notebook from the top (the connection must be reopened to see a rebuilt file).

Use the project's `.venv` as the kernel. All times below are shown in Warsaw local time.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
con = duckdb.connect(ROOT / "data" / "warehouse.duckdb", read_only=True)
con.execute("SET TimeZone = 'Europe/Warsaw'")

## What's in the warehouse

In [ ]:
con.execute("""
    SELECT 'book_snapshots' AS table_name, count(*) AS n_rows,
           min(fetched_at) AS first, max(fetched_at) AS last FROM book_snapshots
    UNION ALL SELECT 'market_status', count(*), min(fetched_at), max(fetched_at) FROM market_status
    UNION ALL SELECT 'metar_obs', count(*), min(first_seen_at), max(first_seen_at) FROM metar_obs
""").df()

In [ ]:
con.execute("""
    SELECT target_date, title, first_seen_at, last_seen_at,
           (SELECT count(*) FROM brackets b WHERE b.event_id = e.event_id) AS n_brackets
    FROM events e
    ORDER BY target_date
""").df()

## Market-implied distribution (latest poll)

Mid prices normalized to sum to 1 across each event's brackets.

In [ ]:
latest = con.execute("""
    SELECT target_date::VARCHAR AS target_date, label, lo_c,
           best_bid::DOUBLE AS bid, best_ask::DOUBLE AS ask,
           mid::DOUBLE AS mid, implied_prob
    FROM market_implied JOIN book_top USING (fetched_at, target_date, label, lo_c, hi_c, mid)
    WHERE fetched_at = (SELECT max(fetched_at) FROM market_implied)
    ORDER BY target_date, lo_c NULLS FIRST
""").df()

dates = sorted(latest["target_date"].unique())
fig, axes = plt.subplots(1, len(dates), figsize=(5 * len(dates), 3.5), sharey=True, squeeze=False)
for ax, d in zip(axes[0], dates, strict=True):
    day = latest[latest["target_date"] == d]
    ax.bar(day["label"], day["implied_prob"])
    ax.set_title(str(d))
    ax.tick_params(axis="x", rotation=60)
axes[0][0].set_ylabel("implied probability")
fig.tight_layout()
latest

## Price history of one event

Mid price of each bracket over time. Pick any `target_date` from the table above.

In [ ]:
TARGET = max(dates)  # the furthest-out event; or any date, e.g. "2026-11-03"

history = con.execute(
    """
    SELECT fetched_at, label, lo_c, mid::DOUBLE AS mid
    FROM book_top
    WHERE target_date = ?
    ORDER BY fetched_at
""",
    [TARGET],
).df()

wide = history.pivot_table(index="fetched_at", columns="label", values="mid")
order = history.drop_duplicates("label").sort_values("lo_c", na_position="first")["label"]
wide = wide[[c for c in order if c in wide and wide[c].max() >= 0.02]]  # only relevant ones

ax = wide.plot(figsize=(10, 4), title=f"Mid price per bracket, event {TARGET}")
ax.set_ylabel("mid price")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))

## Order book depth of the favourite bracket (latest poll)

In [ ]:
depth = con.execute(
    """
    WITH fav AS (
        SELECT * FROM market_implied
        WHERE target_date = ? AND fetched_at = (SELECT max(fetched_at) FROM market_implied)
        ORDER BY implied_prob DESC LIMIT 1
    ), snap AS (
        SELECT s.* FROM book_snapshots s
        JOIN brackets b ON b.yes_token_id = s.token_id
        JOIN events e ON e.event_id = b.event_id
        JOIN fav ON fav.label = b.label AND fav.target_date = e.target_date
                AND fav.fetched_at = s.fetched_at
    )
    SELECT 'bid' AS side, l.price::DOUBLE AS price, l.size::DOUBLE AS size
    FROM snap, unnest(snap.bids) AS t(l)
    UNION ALL
    SELECT 'ask', l.price::DOUBLE, l.size::DOUBLE
    FROM snap, unnest(snap.asks) AS t(l)
    ORDER BY price DESC
""",
    [TARGET],
).df()
depth[(depth["price"] > 0.05) & (depth["price"] < 0.95)]

## METAR temperatures at EPWA

Each distinct report, with the time it was observed. `metar_daily` is a *preliminary* daily
max (routine reports only, latest version of each); the real resolution logic comes later.

In [ ]:
obs = con.execute("""
    SELECT obs_time, report_type, temp_body_c, is_correction, raw
    FROM metar_obs
    ORDER BY obs_time
""").df()

routine = obs[obs["report_type"] != "SPECI"]
ax = routine.plot(
    x="obs_time",
    y="temp_body_c",
    drawstyle="steps-post",
    figsize=(10, 3),
    legend=False,
    title="EPWA temperature (whole °C)",
)
ax.set_ylabel("°C")

con.execute("SELECT * FROM metar_daily ORDER BY local_date").df()